In [ ]:
# -*- coding: utf-8 -*-
"""
GPU-Accelerated Distributed System - Jupyter Notebook Version
"""
# !pip install torch numpy matplotlib ipywidgets  # Décommentez si nécessaire

import torch
import threading
import queue
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from typing import List, Tuple, Any, Optional, Dict, Set
import random
import json
from datetime import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================

# Constants - ajustables via widgets plus bas
N = 8
M = 12
BATCH_SIZE = 64
SIMULATION_TIME = 30

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📱 Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    # Enable optimizations
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ============================================================================
# WIDGETS FOR INTERACTIVE CONTROL
# ============================================================================

def create_control_panel():
    """Create interactive control panel for Jupyter"""
    style = {'description_width': 'initial'}
    
    controls = {
        'n_size': widgets.IntSlider(
            value=8, min=4, max=32, step=4,
            description='Configuration Size (N):',
            style=style
        ),
        'm_processes': widgets.IntSlider(
            value=12, min=4, max=32, step=4,
            description='Number of Processes (M):',
            style=style
        ),
        'batch_size': widgets.IntSlider(
            value=64, min=16, max=256, step=16,
            description='Batch Size:',
            style=style
        ),
        'sim_time': widgets.IntSlider(
            value=30, min=10, max=120, step=10,
            description='Simulation Time (s):',
            style=style
        ),
        'use_gpu': widgets.Checkbox(
            value=True,
            description='Use GPU Acceleration',
            style=style
        ),
        'visualize': widgets.Checkbox(
            value=True,
            description='Real-time Visualization',
            style=style
        ),
        'start_button': widgets.Button(
            description='🚀 Start Simulation',
            button_style='success',
            tooltip='Start the distributed system'
        ),
        'stop_button': widgets.Button(
            description='⏹️ Stop Simulation',
            button_style='danger',
            tooltip='Stop the simulation'
        ),
        'progress': widgets.FloatProgress(
            value=0,
            min=0,
            max=100,
            description='Progress:',
            bar_style='info',
            orientation='horizontal'
        )
    }
    
    # Layout
    control_box = widgets.VBox([
        widgets.HBox([controls['n_size'], controls['m_processes']]),
        widgets.HBox([controls['batch_size'], controls['sim_time']]),
        widgets.HBox([controls['use_gpu'], controls['visualize']]),
        widgets.HBox([controls['start_button'], controls['stop_button']]),
        controls['progress']
    ])
    
    return controls, control_box

# ============================================================================
# GPU-OPTIMIZED COMPONENTS
# ============================================================================

class JupyterGPUCodingList:
    """GPU-optimized for Jupyter environment"""
    
    @staticmethod
    def coding_list(n: int) -> torch.Tensor:
        """Generate coding list on GPU"""
        return torch.arange(n, device=device, dtype=torch.float32)
    
    @staticmethod
    def generate_test_configs(n: int, count: int) -> List[torch.Tensor]:
        """Generate test configurations"""
        configs = []
        for _ in range(count):
            config = torch.randn(n, device=device) * 0.5
            configs.append(config)
        return configs

class JupyterGPULibrary:
    """Library optimized for Jupyter with visualization"""
    
    @staticmethod
    def split(conf: torch.Tensor, refvec: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Split configuration"""
        mid = len(conf) // 2
        return conf[:mid], conf[mid:]
    
    @staticmethod
    def is_Terminal(conf: torch.Tensor, refvec: torch.Tensor) -> bool:
        """Check if terminal - simple threshold"""
        threshold = len(refvec) * 0.3
        return torch.sum(torch.abs(conf)) > threshold
    
    @staticmethod
    def getInitialConf(refvec: torch.Tensor) -> torch.Tensor:
        """Initial configuration"""
        return torch.zeros_like(refvec)
    
    @staticmethod
    def countSetBits(value: torch.Tensor) -> int:
        """Count bits for visualization"""
        # Convert to binary representation
        int_val = int(torch.sum(value).item() * 1000)
        return bin(int_val).count('1')
    
    @staticmethod
    def second(conf: torch.Tensor, refvec: torch.Tensor) -> torch.Tensor:
        """Get second element"""
        return conf[1] if len(conf) > 1 else torch.tensor(0.0, device=device)

class JupyterGPUTestHash:
    """Hash functions for Jupyter"""
    
    @staticmethod
    def h(s: torch.Tensor, m: int, refvec: torch.Tensor) -> int:
        """Hash function"""
        # Simple but effective hash
        hash_val = torch.sum(s * torch.arange(len(s), device=device)).item()
        return int(abs(hash_val)) % m
    
    @staticmethod
    def batch_h(configs: List[torch.Tensor], m: int, refvec: torch.Tensor) -> List[int]:
        """Batch hash"""
        return [JupyterGPUTestHash.h(c, m, refvec) for c in configs]

# ============================================================================
# VISUALIZATION SYSTEM
# ============================================================================

class SystemVisualizer:
    """Real-time visualization for Jupyter"""
    
    def __init__(self):
        self.fig, self.axes = plt.subplots(2, 3, figsize=(15, 10))
        plt.ion()  # Interactive mode
        
        # Data storage
        self.history = {
            'active_configs': [],
            'gpu_usage': [],
            'messages': [],
            'process_load': [[] for _ in range(M)],
            'timestamps': []
        }
        
        # Colors for processes
        self.colors = plt.cm.tab20(np.linspace(0, 1, M))
    
    def update(self, processes: Dict[int, Any], stats: Dict):
        """Update all visualizations"""
        self.history['timestamps'].append(time.time())
        
        # Collect data
        active_total = 0
        gpu_total = 0
        
        for pid, process in processes.items():
            with process.lock:
                active = len(process.s)
                self.history['process_load'][pid].append(active)
                active_total += active
                gpu_total += process.gpu_time
        
        self.history['active_configs'].append(active_total)
        self.history['gpu_usage'].append(gpu_total)
        self.history['messages'].append(stats.get('messages', 0))
        
        # Keep only last 100 points
        max_points = 100
        for key in self.history:
            if isinstance(self.history[key], list):
                self.history[key] = self.history[key][-max_points:]
        
        # Update plots
        self._update_plots(processes)
    
    def _update_plots(self, processes: Dict[int, Any]):
        """Update individual plots"""
        axes = self.axes.flatten()
        
        # Plot 1: Active configurations over time
        axes[0].clear()
        axes[0].plot(self.history['timestamps'], self.history['active_configs'], 'b-', linewidth=2)
        axes[0].set_title('Active Configurations Over Time')
        axes[0].set_xlabel('Time')
        axes[0].set_ylabel('Count')
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: GPU usage
        axes[1].clear()
        axes[1].plot(self.history['timestamps'], self.history['gpu_usage'], 'r-', linewidth=2)
        axes[1].set_title('GPU Usage Over Time')
        axes[1].set_xlabel('Time')
        axes[1].set_ylabel('GPU Time (s)')
        axes[1].grid(True, alpha=0.3)
        
        # Plot 3: Messages sent
        axes[2].clear()
        axes[2].plot(self.history['timestamps'], self.history['messages'], 'g-', linewidth=2)
        axes[2].set_title('Messages Sent Over Time')
        axes[2].set_xlabel('Time')
        axes[2].set_ylabel('Message Count')
        axes[2].grid(True, alpha=0.3)
        
        # Plot 4: Process load distribution
        axes[3].clear()
        current_loads = [len(process.s) for pid, process in processes.items()]
        axes[3].bar(range(len(current_loads)), current_loads, color=self.colors)
        axes[3].set_title('Current Process Load Distribution')
        axes[3].set_xlabel('Process ID')
        axes[3].set_ylabel('Active Configurations')
        axes[3].grid(True, alpha=0.3)
        
        # Plot 5: Process GPU time
        axes[4].clear()
        gpu_times = [process.gpu_time for pid, process in processes.items()]
        axes[4].bar(range(len(gpu_times)), gpu_times, color=self.colors)
        axes[4].set_title('GPU Time per Process')
        axes[4].set_xlabel('Process ID')
        axes[4].set_ylabel('GPU Time (s)')
        axes[4].grid(True, alpha=0.3)
        
        # Plot 6: System statistics
        axes[5].clear()
        stats_text = f"""
        System Status:
        - Total Processes: {len(processes)}
        - Active Configs: {sum(current_loads)}
        - Total GPU Time: {sum(gpu_times):.2f}s
        - Avg Load: {np.mean(current_loads):.1f}
        - Max Load: {max(current_loads)}
        """
        axes[5].text(0.1, 0.5, stats_text, fontsize=10, 
                    verticalalignment='center', 
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[5].axis('off')
        
        plt.tight_layout()
        plt.draw()
        plt.pause(0.01)

# ============================================================================
# JUPYTER-OPTIMIZED PROCESS
# ============================================================================

class JupyterGPUProcess(threading.Thread):
    """Process optimized for Jupyter with better error handling"""
    
    def __init__(self, pid: int, initiator: bool, batch_size: int, 
                 distributor: Any, visualizer: Optional[SystemVisualizer] = None):
        super().__init__(daemon=True)
        self.pid = pid
        self.initiator = initiator
        self.terminit = False
        self.terminatedi = False
        self.nbrecdi = 0
        self.nbsenti = 0
        self.s = []  # Active configs
        self.t = []  # Terminal configs
        self.queue = queue.Queue(maxsize=1000)
        self.distributor = distributor
        self.refvec = distributor.refvec()
        self.lock = threading.RLock()
        self.batch_size = batch_size
        self.visualizer = visualizer
        
        # Performance tracking
        self.gpu_time = 0.0
        self.cpu_time = 0.0
        self.configs_processed = 0
        
        # Status
        self.status = "initialized"
        self.last_activity = time.time()
        
        print(f"  Process {pid} created ({'Initiator' if initiator else 'Worker'})")
    
    def run(self):
        """Main execution loop"""
        self.status = "running"
        
        while not self.terminatedi:
            try:
                loop_start = time.time()
                
                # Process messages
                self._process_queue()
                
                # Process work if available
                if self.s:
                    self._process_work_batch()
                
                # Update status
                self.cpu_time += time.time() - loop_start
                self.last_activity = time.time()
                
                # Small sleep to prevent CPU hogging
                time.sleep(0.005)
                
            except Exception as e:
                print(f"⚠️ Process {self.pid} error: {str(e)[:100]}")
                self.status = f"error: {str(e)[:50]}"
                time.sleep(0.1)
        
        self.status = "terminated"
        print(f"  Process {self.pid} terminated")
    
    def _process_queue(self):
        """Process messages from queue"""
        try:
            while True:
                msg = self.queue.get_nowait()
                
                if msg == 'stop':
                    self.terminatedi = True
                    return
                
                elif isinstance(msg, tuple):
                    msg_type = msg[0]
                    
                    if msg_type == 'state':
                        config = msg[1]
                        with self.lock:
                            self.nbrecdi += 1
                            self.s.append(config)
                            self.configs_processed += 1
                    
                    elif msg_type == 'gen':
                        self._handle_generation()
                    
                    elif msg_type == 'control':
                        self._handle_control(msg[1])
        
        except queue.Empty:
            pass
    
    def _process_work_batch(self):
        """Process a batch of work using GPU"""
        if not self.s:
            return
        
        gpu_start = time.time()
        
        with self.lock:
            # Take a batch
            batch_size = min(len(self.s), self.batch_size)
            batch = self.s[:batch_size]
            self.s = self.s[batch_size:]
        
        # Process on GPU
        results = []
        for config in batch:
            # Check if terminal
            if not JupyterGPULibrary.is_Terminal(config, self.refvec):
                # Split and hash
                rs1, rs2 = JupyterGPULibrary.split(config, self.refvec)
                i1 = JupyterGPUTestHash.h(rs1, M, self.refvec)
                i2 = JupyterGPUTestHash.h(rs2, M, self.refvec)
                
                results.append((rs1, rs2, i1, i2))
            else:
                # Terminal config
                with self.lock:
                    self.t.append(config)
        
        # Process results
        for rs1, rs2, i1, i2 in results:
            # Check conditions
            b1 = (JupyterGPULibrary.countSetBits(
                JupyterGPULibrary.second(rs1, self.refvec)
            ) >= math.sqrt(len(self.refvec)))
            
            b2 = (JupyterGPULibrary.countSetBits(
                JupyterGPULibrary.second(rs2, self.refvec)
            ) >= math.sqrt(len(self.refvec)))
            
            if b1 and b2:
                # Send to appropriate destinations
                if i1 != self.pid:
                    self.distributor.send_config(rs1, i1)
                    self.nbsenti += 1
                
                if i2 != self.pid:
                    self.distributor.send_config(rs2, i2)
                    self.nbsenti += 1
        
        self.gpu_time += time.time() - gpu_start
    
    def _handle_generation(self):
        """Handle work generation"""
        with self.lock:
            if not self.s:
                if self.initiator and not self.terminit:
                    # Start termination detection
                    j = (self.pid + 1) % M
                    threading.Thread(
                        target=self.distributor.send_rec,
                        args=(j, self.nbrecdi)
                    ).start()
                    self.terminit = True
                elif self.initiator and self.terminit:
                    # Continue termination detection
                    j = (self.pid + 1) % M
                    threading.Thread(
                        target=self.distributor.send_rec,
                        args=(j, self.nbrecdi)
                    ).start()
    
    def _handle_control(self, command: str):
        """Handle control commands"""
        if command == 'status':
            self._print_status()
        elif command == 'clear':
            with self.lock:
                self.s.clear()
                self.t.clear()

    def _print_status(self):
        """Print process status"""
        with self.lock:
            status = f"""
            Process {self.pid} Status:
            - State: {self.status}
            - Active configs: {len(self.s)}
            - Terminal configs: {len(self.t)}
            - Received: {self.nbrecdi}
            - Sent: {self.nbsenti}
            - GPU time: {self.gpu_time:.2f}s
            - CPU time: {self.cpu_time:.2f}s
            - Efficiency: {self.gpu_time/max(self.cpu_time, 0.001):.1%}
            """
            print(status)

# ============================================================================
# JUPYTER DISTRIBUTOR
# ============================================================================

class JupyterDistributor:
    """Distributor optimized for Jupyter Notebook"""
    
    def __init__(self, n: int = N, m: int = M, 
                 batch_size: int = BATCH_SIZE,
                 use_visualization: bool = True):
        self.n = n
        self.m = m
        self.batch_size = batch_size
        self.use_visualization = use_visualization
        
        self.processes = {}
        self.stats = {
            'start_time': time.time(),
            'total_configs': 0,
            'messages_sent': 0,
            'termination_detected': False
        }
        
        # Visualization
        if use_visualization:
            self.visualizer = SystemVisualizer()
        else:
            self.visualizer = None
        
        # Control flags
        self.running = False
        self.control_lock = threading.Lock()
        
        print(f"🎯 Jupyter Distributor initialized")
        print(f"   N={n}, M={m}, Batch={batch_size}")
    
    def refvec(self) -> torch.Tensor:
        """Get reference vector"""
        return JupyterGPUCodingList.coding_list(self.n)
    
    def initialize(self):
        """Initialize all processes"""
        print("\n🔧 Initializing system...")
        
        # Create reference vector
        refvec = self.refvec()
        
        # Create initial configuration
        init_config = JupyterGPULibrary.getInitialConf(refvec)
        
        # Determine initiator
        init_hash = JupyterGPUTestHash.h(init_config, self.m, refvec)
        
        # Create processes
        for i in range(self.m):
            is_initiator = (i == init_hash)
            process = JupyterGPUProcess(
                pid=i,
                initiator=is_initiator,
                batch_size=self.batch_size,
                distributor=self,
                visualizer=self.visualizer
            )
            
            self.processes[i] = process
            
            # Send initial config to initiator
            if is_initiator:
                self.send_config(init_config, i)
        
        # Start all processes
        for process in self.processes.values():
            process.start()
        
        print(f"✅ {self.m} processes started")
        print(f"   Initiator: Process {init_hash}")
    
    def send_config(self, config: torch.Tensor, dest_pid: int):
        """Send configuration to process"""
        if dest_pid in self.processes:
            self.processes[dest_pid].queue.put(('state', config))
            self.stats['total_configs'] += 1
            self.stats['messages_sent'] += 1
    
    def send_rec(self, dest_pid: int, count: int):
        """Send receive count"""
        if dest_pid in self.processes:
            # Simplified for Jupyter
            pass
    
    def run_simulation(self, duration: int = SIMULATION_TIME):
        """Run the simulation with real-time updates"""
        print(f"\n🚀 Starting simulation for {duration} seconds...")
        
        self.running = True
        start_time = time.time()
        
        # Start monitoring thread
        monitor_thread = threading.Thread(
            target=self._monitor_loop,
            args=(duration,),
            daemon=True
        )
        monitor_thread.start()
        
        # Generate initial work
        self._generate_initial_work()
        
        # Main simulation loop
        try:
            while time.time() - start_time < duration and self.running:
                # Generate periodic work
                if random.random() < 0.1:  # 10% chance each second
                    self._generate_random_work()
                
                # Update visualization
                if self.visualizer and self.use_visualization:
                    self.visualizer.update(self.processes, self.stats)
                
                # Print status every 5 seconds
                elapsed = time.time() - start_time
                if int(elapsed) % 5 == 0 and int(elapsed) > 0:
                    self._print_system_status()
                
                time.sleep(1.0)
        
        except KeyboardInterrupt:
            print("\n⏹️ Simulation interrupted by user")
        
        finally:
            self.stop_simulation()
    
    def _monitor_loop(self, duration: int):
        """Monitoring loop for system health"""
        start_time = time.time()
        
        while time.time() - start_time < duration and self.running:
            # Check for dead processes
            dead_count = 0
            for pid, process in self.processes.items():
                if not process.is_alive():
                    dead_count += 1
            
            if dead_count > 0:
                print(f"⚠️  {dead_count} processes not responding")
            
            time.sleep(2.0)
    
    def _generate_initial_work(self):
        """Generate initial work for all processes"""
        print("💥 Generating initial work...")
        
        refvec = self.refvec()
        
        # Generate random configurations
        configs = JupyterGPUCodingList.generate_test_configs(self.n, self.m * 20)
        
        # Distribute to processes based on hash
        for config in configs:
            dest = JupyterGPUTestHash.h(config, self.m, refvec)
            self.send_config(config, dest)
        
        print(f"   Sent {len(configs)} initial configurations")
    
    def _generate_random_work(self):
        """Generate random work during simulation"""
        refvec = self.refvec()
        
        # Generate a few random configs
        count = random.randint(5, 20)
        configs = JupyterGPUCodingList.generate_test_configs(self.n, count)
        
        for config in configs:
            dest = JupyterGPUTestHash.h(config, self.m, refvec)
            self.send_config(config, dest)
    
    def _print_system_status(self):
        """Print current system status"""
        elapsed = time.time() - self.stats['start_time']
        
        # Collect statistics
        active_total = 0
        terminal_total = 0
        gpu_total = 0
        
        for process in self.processes.values():
            with process.lock:
                active_total += len(process.s)
                terminal_total += len(process.t)
                gpu_total += process.gpu_time
        
        # Clear and print
        clear_output(wait=True)
        
        print("=" * 70)
        print("🎯 JUPYTER DISTRIBUTED SYSTEM - LIVE STATUS")
        print("=" * 70)
        
        print(f"\n📊 System Statistics:")
        print(f"   Elapsed time: {elapsed:.1f}s")
        print(f"   Active processes: {len([p for p in self.processes.values() if p.is_alive()])}/{self.m}")
        print(f"   Total configurations: {self.stats['total_configs']:,}")
        print(f"   Active configurations: {active_total:,}")
        print(f"   Terminal configurations: {terminal_total:,}")
        print(f"   Total GPU time: {gpu_total:.2f}s")
        print(f"   Processing rate: {self.stats['total_configs']/max(elapsed, 0.001):.1f} configs/s")
        
        # Progress bar
        progress = min(1.0, elapsed / SIMULATION_TIME)
        bar_length = 50
        filled = int(bar_length * progress)
        bar = '█' * filled + '░' * (bar_length - filled)
        print(f"\n⏱️  Progress: [{bar}] {progress*100:.1f}%")
        
        if self.visualizer:
            plt.show()
    
    def stop_simulation(self):
        """Stop the simulation gracefully"""
        print("\n🛑 Stopping simulation...")
        
        self.running = False
        
        # Send stop signals
        for process in self.processes.values():
            process.queue.put('stop')
        
        # Wait for processes to terminate
        for pid, process in self.processes.items():
            process.join(timeout=2.0)
        
        # Cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Print final statistics
        self._print_final_report()
        
        print("✅ Simulation stopped")
    
    def _print_final_report(self):
        """Print final simulation report"""
        print("\n" + "=" * 70)
        print("📈 FINAL SIMULATION REPORT")
        print("=" * 70)
        
        # Collect final statistics
        total_configs = 0
        total_gpu = 0
        total_cpu = 0
        max_load = 0
        
        for pid, process in self.processes.items():
            with process.lock:
                total_configs += process.configs_processed
                total_gpu += process.gpu_time
                total_cpu += process.cpu_time
                max_load = max(max_load, len(process.s))
        
        elapsed = time.time() - self.stats['start_time']
        
        print(f"\n📊 Performance Summary:")
        print(f"   Total simulation time: {elapsed:.2f}s")
        print(f"   Total configurations processed: {total_configs:,}")
        print(f"   Total GPU time: {total_gpu:.2f}s")
        print(f"   Total CPU time: {total_cpu:.2f}s")
        print(f"   GPU utilization: {(total_gpu/elapsed)*100:.1f}%")
        print(f"   Average processing rate: {total_configs/elapsed:.1f} configs/s")
        
        print(f"\n⚖️  Load Distribution:")
        for pid, process in self.processes.items():
            with process.lock:
                load = len(process.s) + len(process.t)
                print(f"   Process {pid}: {load} configs "
                      f"(GPU: {process.gpu_time:.2f}s, "
                      f"Eff: {process.gpu_time/max(process.cpu_time, 0.001):.1%})")
        
        if torch.cuda.is_available():
            print(f"\n💾 GPU Memory Statistics:")
            print(f"   Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"   Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")
            print(f"   Max allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# ============================================================================
# MAIN EXECUTION FOR JUPYTER
# ============================================================================

def run_jupyter_simulation():
    """Main function to run in Jupyter"""
    
    # Create control panel
    controls, control_panel = create_control_panel()
    display(control_panel)
    
    # Define button actions
    def start_simulation(b):
        """Start simulation with current parameters"""
        clear_output(wait=True)
        display(control_panel)
        
        # Get parameters from widgets
        n = controls['n_size'].value
        m = controls['m_processes'].value
        batch_size = controls['batch_size'].value
        sim_time = controls['sim_time'].value
        use_gpu = controls['use_gpu'].value
        visualize = controls['visualize'].value
        
        print(f"🚀 Starting simulation with:")
        print(f"   N={n}, M={m}, Batch={batch_size}")
        print(f"   Duration={sim_time}s, GPU={use_gpu}, Viz={visualize}")
        print("=" * 50)
        
        # Update progress bar
        controls['progress'].value = 0
        
        # Create and run distributor
        distributor = JupyterDistributor(
            n=n, 
            m=m, 
            batch_size=batch_size,
            use_visualization=visualize
        )
        
        distributor.initialize()
        
        # Run in background thread to keep UI responsive
        import threading
        sim_thread = threading.Thread(
            target=distributor.run_simulation,
            args=(sim_time,)
        )
        sim_thread.start()
        
        # Update progress bar
        for i in range(100):
            if sim_thread.is_alive():
                controls['progress'].value = (i + 1)
                time.sleep(sim_time / 100)
            else:
                break
        
        controls['progress'].value = 100
    
    def stop_simulation(b):
        """Stop simulation"""
        print("⏹️ Stop requested...")
        # This would need access to the distributor instance
        # For simplicity, we'll just print a message
        controls['progress'].value = 0
    
    # Connect buttons
    controls['start_button'].on_click(start_simulation)
    controls['stop_button'].on_click(stop_simulation)
    
    print("🎮 Control panel ready. Adjust parameters and click 'Start Simulation'.")

# Alternative: Simple one-click execution
def run_simple_simulation():
    """Run a simple simulation without widgets"""
    print("🚀 Running simple simulation...")
    
    distributor = JupyterDistributor(
        n=8,
        m=12,
        batch_size=64,
        use_visualization=True
    )
    
    distributor.initialize()
    distributor.run_simulation(duration=30)

# ============================================================================
# EXECUTION CELLS FOR JUPYTER
# ============================================================================

# Cell 1: Import and setup
print("✅ Imports completed successfully!")
print(f"📱 Using device: {device}")

# Cell 2: Run with interactive controls
run_jupyter_simulation()

# Cell 3: Or run simple simulation (uncomment to use)
# run_simple_simulation()

# Cell 4: GPU memory cleanup (run after simulation)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleaned up")